[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E5%8D%94%E4%BD%9C%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92_%E8%81%B7%E5%A0%B4%E5%AF%A6%E6%88%B0%E7%AF%87/01_%E6%A5%AD%E5%8B%99_%E6%BD%9B%E5%9C%A8%E5%AE%A2%E6%88%B6%E6%88%90%E4%BA%A4%E9%A0%90%E6%B8%AC/%E6%BD%9B%E5%9C%A8%E5%AE%A2%E6%88%B6%E6%88%90%E4%BA%A4%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87.ipynb)

# 💼 範例 01：B2B 業務實戰—潛在客戶成交機率預測 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本教材專為**「B2B 業務、業務主管與個人接案族（無程式背景）」**設計。
採用**「一段保證可執行的 Python 程式碼 ＋ 一段 AI 伴學互動提問指引」**的雙軌架構：
- 每一段程式碼都經過嚴謹測試，保證直接點擊即可順暢執行。
- 執行完程式碼後，**下方的文字儲存格**會以業務實務白話解釋程式在做什麼，並提供專屬的 **AI 提問 Prompt（提示詞）**，你可以複製貼給 Copilot、ChatGPT 或 Claude，讓 AI 替你深度解說商業決策邏輯！

---

## 📊 一、業務資料集介紹 (`leads.csv`)

本數據集記錄了某企業過去接洽之 2,000 筆商務詢價潛在客戶（Leads）條件與最終是否成功簽約成交：

| 欄位名稱 | 資料型態 | 數值範例 / 類別 | 商業業務實務意涵 |
| :--- | :--- | :--- | :--- |
| **客戶產業** | 類別型 (`object`) | 科技資訊、傳統製造、批發零售、醫療生技、金融保險 | 客戶所屬行業領域，影響軟硬體採購導入意願 |
| **企業規模員工人數** | 數值型 (`int64`) | 50 (人，範圍 15~1500 人) | 客戶組織規模，決定預算上限與審核流程複雜度 |
| **詢價預算_TWD** | 數值型 (`int64`) | 350,000 (元，範圍 2 萬~300 萬元) | 客戶提出之初步採購預算 |
| **歷史接洽次數** | 數值型 (`int64`) | 3 (次，範圍 1~7 次) | 業務已拜訪或電話會議溝通次數 |
| **窗口職級** | 類別型 (`object`) | 基層專員、採購主管、部門經理、總經理/負責人 | 對接窗口是否有直接採購拍板決策權 |
| **是否索取產品試用** | 類別型 (`object`) | 是、否 | 客戶是否主動開通試用帳號或樣品測試（高意向指標） |
| **是否參加線上說明會** | 類別型 (`object`) | 是、否 | 是否曾參與產品 Demo 或網路研討會（高互動指標） |
| **預估決策天數** | 數值型 (`int64`) | 30 (天) | 客戶預期採購流程推進時長 |
| **是否成交 (目標 y)** | 類別型 (`object`) | **「是」 / 「否」** | **預測目標：該筆詢價最後是否成功簽約付款** |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與繁中字型**：確保圖表顯示中文，避免方塊亂碼 (`□□`)。
2. **資料載入與探索 (EDA)**：檢視預算分佈、產業比例與成交率。
3. **特徵關聯性與視覺化**：分析試用與窗口職級對成交率的實質影響。
4. **特徵工程與 Pipeline 建置**：8:2 切分訓練與測試集，防止業務資料洩漏。
5. **模型選型與 5 折交叉驗證**：比對「邏輯回歸 Baseline」與「隨機森林分類器」。
6. **測試集大考與業務跟進策略**：解讀混淆矩陣，輸出模型並模擬新詢價即時評估。

---  
### 【步驟 0】：環境準備與繁體中文字型設定

In [ ]:
# [步驟 0 程式碼] 載入套件與設定繁體中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Heiti TC", "Microsoft JhengHei", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！準備進入 B2B 潛在客戶成交率分析。")

#### 💬【步驟 0 學習指引與 AI 提問 Prompt】
- **這段程式在做什麼？**
  載入 Pandas 與 Matplotlib 等套件，並設定繁中字型支援，徹底解決資料科學繪圖時中文變亂碼方塊的經典問題。

💡 **提問 Prompt**：
> 「我是非工程師的業務主管，請用白話告訴我為什麼 Python 繪圖預設無法直接顯示中文？這段程式是如何幫我自動修正的？」

---  
### 【步驟 1】：載入業務詢價資料集與基本結構探索 (EDA)

In [ ]:
# [步驟 1 程式碼] 讀取資料與結構探索
df = pd.read_csv("leads.csv")

print(f"=== 資料規模：共 {df.shape[0]} 筆潛在客戶紀錄，{df.shape[1]} 個欄位 ===")
print("\n--- 資料欄位型態與缺失值檢視 ---")
df.info()

print("\n--- 前 5 筆潛在客戶樣本 ---")
display(df.head())

print("\n--- 最終成交比例分佈 ---")
print(df["是否成交"].value_counts(normalize=True).round(3) * 100)

---  
### 【步驟 2】：特徵關聯性與視覺化圖表 (EDA)
業務最關心的問題：**「索取試用」與「窗口職級」是否對最終能否簽約成交起決定性作用？**

In [ ]:
# [步驟 2 程式碼] 試用與窗口職級對成交率的交叉視覺化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 圖 1: 是否索取試用與成交率
trial_rate = df.groupby("是否索取產品試用")["是否成交"].apply(lambda s: (s == "是").mean() * 100)
trial_rate.plot(kind="bar", ax=axes[0], color="#3182ce")
axes[0].set_title("索取產品試用與成交機率 (%) 關聯", fontsize=12)
axes[0].set_ylabel("成交率 (%)")
axes[0].grid(axis="y", linestyle="--", alpha=0.7)

# 圖 2: 窗口職級與成交人數分佈
sns.countplot(data=df, x="窗口職級", hue="是否成交", palette=["#e53e3e", "#38a169"], ax=axes[1])
axes[1].set_title("對接窗口職級與成交人數分佈", fontsize=12)
axes[1].set_ylabel("客戶數量")
axes[1].legend(title="是否成交")
axes[1].grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

#### 💬【步驟 2 學習指引與 AI 提問 Prompt】
- **這段程式在做什麼？**
  圖表直接揭示業務實務真相：有索取試用的客戶成交率高達 70% 以上；直接對接到「部門經理」或「總經理」的成交人數遠多於對接基層專員，量化印證「找對決策人」是業務成交的核心動能。

---  
### 【步驟 3】：特徵工程與 Pipeline 建置 (嚴防資料洩漏)

In [ ]:
# [步驟 3 程式碼] 資料切分與前處理 Pipeline
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=["是否成交"])
y = (df["是否成交"] == "是").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["企業規模員工人數", "詢價預算_TWD", "歷史接洽次數", "預估決策天數"]
categorical_features = ["客戶產業", "窗口職級", "是否索取產品試用", "是否參加線上說明會"]

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
])

print(f"✅ 訓練集筆數: {X_train.shape[0]} 筆")
print(f"✅ 測試集筆數: {X_test.shape[0]} 筆 (封存驗證)")

---  
### 【步驟 4】：模型選型與 5 折交叉驗證 (邏輯回歸 vs 隨機森林)

In [ ]:
# [步驟 4 程式碼] 建立 Pipeline 與 5 折交叉驗證
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

pipeline_lr = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(random_state=42))
])

pipeline_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42))
])

cv_lr = cross_val_score(pipeline_lr, X_train, y_train, cv=5, scoring="accuracy")
cv_rf = cross_val_score(pipeline_rf, X_train, y_train, cv=5, scoring="accuracy")

print("=== 5 折交叉驗證平均準確率比對 ===")
print(f"🔹 基準模型 (邏輯回歸): {cv_lr.mean():.4f} (標準差: {cv_lr.std():.4f})")
print(f"🌲 集成模型 (隨機森林): {cv_rf.mean():.4f} (標準差: {cv_rf.std():.4f})")

best_pipeline = pipeline_rf
best_pipeline.fit(X_train, y_train)
print("\n✅ 隨機森林模型訓練完成！")

---  
### 【步驟 5】：測試集大考、混淆矩陣與特徵重要性分析

In [ ]:
# [步驟 5 程式碼] 測試集推論與評估圖表
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

print("=== 業務成交預測：測試集分類評估報告 ===")
print(classification_report(y_test, y_pred, target_names=["未成交(0)", "成交(1)"]))
print(f"🎯 ROC-AUC 分數: {roc_auc_score(y_test, y_prob):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["未成交", "成交"])
disp.plot(cmap="Blues", ax=axes[0])
axes[0].set_title("成交預測混淆矩陣 (Confusion Matrix)", fontsize=12)

rf_model = best_pipeline.named_steps["classifier"]
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
ohe_cols = list(ohe.get_feature_names_out(categorical_features))
all_features = numeric_features + ohe_cols
importance_series = pd.Series(rf_model.feature_importances_, index=all_features).sort_values(ascending=False)

importance_series.head(7).plot(kind="barh", ax=axes[1], color="#2b6cb0").invert_yaxis()
axes[1].set_title("影響客戶成交關鍵特徵重要性 (Top 7)", fontsize=12)
axes[1].set_xlabel("重要性權重")

plt.tight_layout()
plt.show()

---  
### 【步驟 6】：實務新詢價即時推論 (Inference) 與模型封裝

In [ ]:
# [步驟 6 程式碼] 即時推論與模型保存
import joblib

case_samples = pd.DataFrame([
    {
        "客戶產業": "科技資訊",
        "企業規模員工人數": 150,
        "詢價預算_TWD": 650000,
        "歷史接洽次數": 3,
        "窗口職級": "部門經理",
        "是否索取產品試用": "是",
        "是否參加線上說明會": "是",
        "預估決策天數": 25
    },
    {
        "客戶產業": "傳統製造",
        "企業規模員工人數": 20,
        "詢價預算_TWD": 50000,
        "歷史接洽次數": 1,
        "窗口職級": "基層專員",
        "是否索取產品試用": "否",
        "是否參加線上說明會": "否",
        "預估決策天數": 75
    }
])

probs = best_pipeline.predict_proba(case_samples)[:, 1]

print("=== 潛在客戶成交率即時試算 ===")
print(f"客戶 A (高預算、有試用、經理決策) ➜ 預估成交機率: {probs[0]*100:.1f}% 🔥 [優先級：最高]")
print(f"客戶 B (低預算、無試用、專員接洽、週期長) ➜ 預估成交機率: {probs[1]*100:.1f}% ❄️ [優先級：一般觀望]")

joblib.dump(best_pipeline, "model.joblib")
print("\n✅ 模型已成功匯出為 model.joblib！")
print("🎉 恭喜完成！現在您可以複製 README.md 中的專屬 Prompt，請 AI 替你生成獨立的 Tkinter 桌面成交試算 App！")